## Imports

In [95]:
import os
import hashlib
import chardet
import pandas as pd
import boto3
import yaml

def read_yaml_file(path, file):
    # reading credentials files
    with open(f"{os.path.join(path, file)}") as f:
        try:
            content = yaml.safe_load(f)
        except yaml.YAMLError as e:
            raise e
    
    return content

CONFIG_PATH = os.path.join("..", "src", "config")

In [96]:
credentials_config = read_yaml_file(
    path=CONFIG_PATH,
    file="credentials.yaml"
)

DIR_PATH = os.path.join('..', 'data', 'raw')
PREPROCESSED_PATH = os.path.join('..', 'data', 'preprocessed')

## Processing raw data

In [97]:
if credentials_config["S3"] != "YOUR_S3_BUCKET_URL":
    s3 = boto3.client(
        "s3",
        aws_access_key_id=credentials_config["AWS_ACCESS_KEY"],
        aws_secret_access_key=credentials_config["AWS_SECRET_KEY"]
    )

In [98]:
def hash_name(name):
    return hashlib.sha256(name.encode()).hexdigest()

In [99]:
# Removing 'NA' from default list of null values
NA_VALUES = [
    "", 
    "#N/A", 
    "#N/A N/A", 
    "#NA", 
    "-1.#IND", 
    "-1.#QNAN", 
    "-NaN", 
    "-nan", 
    "1.#IND", 
    "1.#QNAN", 
    "<NA>", 
    "N/A", 
    "NULL", 
    "NaN", 
    "n/a", 
    "nan", 
    "null"
    ]

In [100]:
for filename in os.listdir(DIR_PATH):
    new_file_name = f'preprocessed_{filename}'
    PREPROCESSED_FILENAME = os.path.join(PREPROCESSED_PATH, new_file_name)
    FILE_PATH = os.path.join(DIR_PATH, filename)
    if os.path.exists(PREPROCESSED_FILENAME):
        continue
    
    with open(FILE_PATH, 'rb') as f:
        result = chardet.detect(f.read())
    df = pd.read_csv(
        FILE_PATH, 
        na_values=NA_VALUES, 
        keep_default_na=False,
        encoding=result['encoding']
        )
    full_name = pd.concat([df.filter(regex=r'(?i)(last name|surname)'), (df.filter(regex=r'(?i)(first name)'))], axis=1)
    cols_to_remove = [full_name.columns[0], full_name.columns[1]]
    temp = full_name[full_name.columns[0]] + ' ' + full_name[full_name.columns[1]]
    df['ID'] = temp.apply(hash_name)

    df = df.drop(columns=cols_to_remove)

    os.makedirs(PREPROCESSED_PATH, exist_ok=True)
    df.to_csv(PREPROCESSED_FILENAME, index=False)

    if credentials_config["S3"] != "YOUR_S3_BUCKET_URL":
        s3.upload_file(
            PREPROCESSED_FILENAME,
            credentials_config["S3"],
            f'preprocessed_{filename}'
        )